<a href="https://colab.research.google.com/github/cy6erlizard/landsat-lake-clarity/blob/main/notebooks/02_select_lakes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 - Choose the target lakes, and measure the ceiling

Region: **Northern Lower Michigan** (clear glacial kettle lakes, same optical regime as the
New Hampshire reference lakes, with decades of Cooperative Lakes Monitoring Program Secchi).

**The gate.** One large lake and one small one, both with enough FIELD July-years to support a
client-style July-annual-mean validation. Selection is on distinct years with a July field
Secchi reading in the Water Quality Portal, NOT on coincident satellite/in-situ matchups.
Matchups require a clean pass within three days of a reading and badly undercount the
achievable validation sample; the client's own r = -0.22 for Squam used the full field record,
not matchups. If no small lake qualifies, `select_target_lakes` raises.

**The ceiling.** The intraclass correlation of log Secchi across the region's lakes: the
fraction of the pooled signal that says nothing about tracking one lake through time. It is the
number that reconciles a published R-squared of 0.637 with a per-lake r of -0.22.

In [1]:
# Cell 1 of 6: repo + Drive
import os, pathlib, subprocess, sys

REPO = "https://github.com/cy6erlizard/landsat-lake-clarity.git"
ROOT = pathlib.Path("/content/landsat-lake-clarity")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT / "src"))  # make lakeclarity importable in the running kernel

from google.colab import drive
drive.mount("/content/drive")
os.environ["LAKECLARITY_DATA"] = "/content/drive/MyDrive/lake-clarity"

Mounted at /content/drive


In [2]:
# Cell 2 of 6: training set (all Michigan) and target region (northern counties)
import pandas as pd
from lakeclarity import config, eda, edi, features, locus, selection, viz, wqp

viz.use_style()

matchups = edi.load_matchups()
lakes = locus.load_lakes()

# Piper, Glines & Rose trained on ALL Wisconsin lakes in AquaSat, not just their
# 127 study lakes. The equivalent is every Michigan lake. Target lakes come from
# the narrower northern-county region and are held out of training entirely.
train_lk = locus.train_lakes(lakes)
region_lk = locus.region_lakes(lakes)
print(f"{len(train_lk):,} {config.TRAIN_REGION_NAME} lakes, "
      f"{len(region_lk):,} {config.REGION_NAME} lakes in LOCUS")

region = matchups[matchups["lagoslakeid"].isin(train_lk["lagoslakeid"])].copy()
region = region[region[config.TARGET].notna()]
region = features.add_time_columns(region)
region = features.add_log_target(region)
print(f"{len(region):,} Secchi matchups on {region.lagoslakeid.nunique():,} lakes (training supply)")
region.to_parquet(config.INTERIM_DIR / "region_matchups.parquet", index=False)

15,905 Michigan lakes, 1,633 Northern Lower Michigan lakes in LOCUS
3,213 Secchi matchups on 716 lakes (training supply)


In [3]:
# Cell 3 of 6: FIELD coverage from the Water Quality Portal (the selection metric)
# Confirm the characteristic-name choice, then pull field Secchi + station coords,
# map each station to its nearest lake, and count distinct July-years per lake.
print(wqp.characteristic_coverage().to_string(index=False), "\n")

field = wqp.fetch_secchi()                 # cached to Drive after first pull
stations = wqp.fetch_stations()            # site coordinates
site_to_lake = wqp.map_sites_to_lakes(stations, lakes)
field_cov = wqp.lake_field_coverage(field, site_to_lake)

site_to_lake.to_parquet(config.INTERIM_DIR / "wqp_site_to_lake.parquet", index=False)
field_cov.to_csv(config.TABLE_DIR / "T04b_field_coverage.csv")
print(f"{len(field):,} field Secchi records, {len(stations):,} stations, "
      f"{len(site_to_lake):,} mapped to a lake")
print(f"lakes with >= {config.MIN_FIELD_JULY_YEARS} field July-years: "
      f"{(field_cov['field_july_years'] >= config.MIN_FIELD_JULY_YEARS).sum()}")
print(field_cov.head(12).to_string())

                 characteristic  n_records         status
       Depth, Secchi disk depth      61512             ok
  Secchi Reservoir Transparency          0 rejected (400)
Water transparency, Secchi disc          0             ok 

60,809 field Secchi records, 941 stations, 694 mapped to a lake
lakes with >= 15 field July-years: 50
             field_july_years  field_july_n  field_year_first  field_year_last  field_secchi_mean  field_all_years
lagoslakeid                                                                                                       
2392                       39           336              1984             2023           2.914499               39
1141                       36           560              1984             2023           4.239576               39
1362                       35           213              1984             2023          10.165428               35
1627                       35           324              1984             2019           

In [ ]:
# Cell 4 of 6: THE CEILING LADDER. Run this before choosing anything.
# ICC is a property of the population, not of nature. Tighten the population from
# the whole US to one optically homogeneous region and the between-lake variance
# that a pooled R2 mostly measures collapses, exposing the temporal signal a
# regional model can actually learn. Compute it at three tiers to show the mechanism.
national = matchups[matchups[config.TARGET].notna()].copy()
national = features.add_log_target(national)

# The homogeneous target region (northern counties), NOT the whole training state.
# This is where between-lake variance is expected to collapse.
homreg = matchups[matchups["lagoslakeid"].isin(region_lk["lagoslakeid"])].copy()
homreg = homreg[homreg[config.TARGET].notna()]
homreg = features.add_log_target(homreg)

tiers = {
    "national (all US)": selection.ceiling_report(national),
    f"{config.TRAIN_REGION_NAME} (state)": selection.ceiling_report(region),
    config.REGION_NAME: selection.ceiling_report(homreg),
}
labels = list(tiers)

print(f"{'':>24}" + "".join(f"{l:>24}" for l in labels))
for k in ["icc", "between_lake_sd_log10", "within_lake_sd_log10", "n_lakes", "n_obs"]:
    print(f"{k:>24}" + "".join(f"{tiers[l][k]:>24,.3f}" for l in labels))

nat, hom = tiers[labels[0]], tiers[labels[-1]]
print(f"\nNational: {nat['pct_variance_between_lakes']:.0f}% of the variance is lakes")
print("differing from one another. That is what a pooled R2 of 0.637 mostly measures,")
print("and it says nothing about tracking one lake through time.")
print(f"\nInside {config.REGION_NAME} the lakes resemble each other, so between-lake")
print(f"variance falls to {hom['pct_variance_between_lakes']:.0f}% and "
      f"{hom['pct_variance_within_lakes']:.0f}% of what remains is genuine")
print("temporal signal, the signal a regional model learns and a national model drowns.")

pd.DataFrame(tiers).to_csv(config.TABLE_DIR / "T03b_icc_ladder.csv")
pd.Series(tiers[config.REGION_NAME]).to_csv(config.TABLE_DIR / "T03_variance_ceiling.csv")

In [5]:
# Cell 5 of 6: the gate
candidates = selection.candidate_table(region, region_lk, field_cov)
candidates.to_csv(config.TABLE_DIR / "T04_candidate_lakes.csv", index=False)
print(candidates.head(15)[["lagoslakeid", "lake_name", "lake_waterarea_ha",
                            "field_july_years", "field_year_first", "field_year_last",
                            "field_secchi_mean", "n_matchups"]].to_string(index=False))

try:
    targets = selection.select_target_lakes(candidates)
    print("\n" + targets.summary())
    pd.Series({"large": targets.ids[0], "small": targets.ids[1]}).to_csv(
        config.TABLE_DIR / "T05_target_lakes.csv")
except selection.NoEligibleLakeError as exc:
    print("\nGATE FAILED:", exc)
    print("\nDo not work around this. Either relax MIN_FIELD_JULY_YEARS with a written")
    print("justification, widen the region, or accept that the small-lake case cannot")
    print("be validated with the data that exists.")
    raise

 lagoslakeid        lake_name  lake_waterarea_ha  field_july_years  field_year_first  field_year_last  field_secchi_mean  n_matchups
      1141.0        Glen Lake        2546.573792                36              1984             2023           4.239576           9
      1627.0      Platte Lake        1025.222497                35              1984             2019           3.884793         212
      1362.0     Higgins Lake        4126.624360                35              1984             2023          10.165428          26
      3270.0     Arbutus Lake         153.210846                34              1988             2023           4.492502           9
    139057.0 Stone Ledge Lake          32.657208                31              1987             2023           2.932450           4
      2855.0        Clam Lake         177.128046                29              1990             2024           4.172169          30
       602.0    Lake Leelanau        3486.426754                29   

In [ ]:
# Cell 6 of 6: figures F6 to F9
# F08 contrasts the two extremes of the ladder: the national distribution against
# the homogeneous target region, where between-lake variance collapses.
for fig, fid, slug in [
    (selection.fig_region_map(candidates, targets), "F06", "region_map"),
    (selection.fig_area_vs_pixels(candidates, targets), "F07", "area_vs_pixels"),
    (selection.fig_variance_decomposition(homreg, national=national,
                                          train_label=config.REGION_NAME),
     "F08", "variance_decomposition"),
    (selection.fig_candidate_timeseries(field, site_to_lake, candidates), "F09", "candidate_timeseries"),
]:
    print("wrote", viz.save(fig, fid, slug).name)